In [3]:
pip install noisereduce


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import numpy as np
import librosa
import cv2
from tensorflow import keras
import noisereduce as nr

model = keras.models.load_model('speech_overlap_detector.h5')

def preprocess_audio_segment(y, sr=22050, img_size=(128, 128)):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_db = (mel_spec_db - np.min(mel_spec_db)) / (np.max(mel_spec_db) - np.min(mel_spec_db))
    mel_spec_db = cv2.resize(mel_spec_db, img_size, interpolation=cv2.INTER_AREA)
    return mel_spec_db

def detect_exact_overlap(audio_file, frame_size=0.5, step_size=0.1, sr=22050, threshold=0.6):
    y, _ = librosa.load(audio_file, sr=sr)
    y = nr.reduce_noise(y=y, sr=sr, stationary=False)
    audio_length = len(y)
    overlap_timestamps = []

    frame_length = int(frame_size * sr)
    step_length = int(step_size * sr)

    for start in range(0, audio_length - frame_length, step_length):
        end = start + frame_length
        audio_frame = y[start:end]

        spectrogram = preprocess_audio_segment(audio_frame, sr=sr)
        spectrogram = np.expand_dims(spectrogram, axis=(0, -1))

        prediction = model.predict(spectrogram, verbose=0)[0][0]
        print(f"Frame {start/sr:.2f} - {end/sr:.2f}, Prediction: {prediction:.2f}")

        if prediction > threshold:
            overlap_start = start / sr
            overlap_end = end / sr
            if overlap_start != overlap_end:  # Skip zero-duration segments
                overlap_timestamps.append((overlap_start, overlap_end))

    return overlap_timestamps

def merge_segments(segments, gap_threshold=0.5):
    if not segments:
        return []
    segments = sorted(segments, key=lambda x: x[0])
    merged = [segments[0]]
    for current in segments[1:]:
        last = merged[-1]
        if current[0] <= last[1] + gap_threshold:
            merged[-1] = (last[0], max(last[1], current[1]))
        else:
            merged.append(current)
    return merged

def filter_short_segments(segments, min_duration=1.0):
    return [seg for seg in segments if (seg[1] - seg[0]) >= min_duration]

def seconds_to_min_sec(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"

def save_timestamps(timestamps, output_file="exact_overlap_timestamps.txt"):
    with open(output_file, "w") as f:
        for start, end in timestamps:
            start_str = seconds_to_min_sec(start)
            end_str = seconds_to_min_sec(end)
            f.write(f"{start_str},{end_str}\n")
    print(f"Timestamps saved to {output_file}")

# Example usage
audio_file = 'B007.wav'
raw_overlap_times = detect_exact_overlap(audio_file, frame_size=0.5, step_size=0.1, threshold=0.6)
merged_overlap_times = merge_segments(raw_overlap_times, gap_threshold=0.5)
final_overlap_times = filter_short_segments(merged_overlap_times, min_duration=1.0)

print("\nFinal Overlap Timestamps:")
for start, end in final_overlap_times:
    start_str = seconds_to_min_sec(start)
    end_str = seconds_to_min_sec(end)
    print(f"Overlap from {start_str} to {end_str}")

save_timestamps(final_overlap_times)

Frame 0.00 - 0.50, Prediction: 0.46
Frame 0.10 - 0.60, Prediction: 0.50
Frame 0.20 - 0.70, Prediction: 0.51
Frame 0.30 - 0.80, Prediction: 0.47
Frame 0.40 - 0.90, Prediction: 0.57
Frame 0.50 - 1.00, Prediction: 0.49
Frame 0.60 - 1.10, Prediction: 0.45
Frame 0.70 - 1.20, Prediction: 0.50
Frame 0.80 - 1.30, Prediction: 0.49


C:\Users\saksh\AppData\Local\Temp\ipykernel_5740\3324831145.py:13: RuntimeWarning: invalid value encountered in divide
  mel_spec_db = (mel_spec_db - np.min(mel_spec_db)) / (np.max(mel_spec_db) - np.min(mel_spec_db))


Frame 0.90 - 1.40, Prediction: 0.53
Frame 1.00 - 1.50, Prediction: 0.53
Frame 1.10 - 1.60, Prediction: 0.53
Frame 1.20 - 1.70, Prediction: 0.53
Frame 1.30 - 1.80, Prediction: 0.57
Frame 1.40 - 1.90, Prediction: 0.62
Frame 1.50 - 2.00, Prediction: 0.58
Frame 1.60 - 2.10, Prediction: 0.52
Frame 1.70 - 2.20, Prediction: 0.54
Frame 1.80 - 2.30, Prediction: 0.54
Frame 1.90 - 2.40, Prediction: 0.51
Frame 2.00 - 2.50, Prediction: 0.45
Frame 2.10 - 2.60, Prediction: 0.51
Frame 2.20 - 2.70, Prediction: 0.57
Frame 2.30 - 2.80, Prediction: 0.53
Frame 2.40 - 2.90, Prediction: 0.57
Frame 2.50 - 3.00, Prediction: 0.51
Frame 2.60 - 3.10, Prediction: 0.51
Frame 2.70 - 3.20, Prediction: 0.51
Frame 2.80 - 3.30, Prediction: 0.46
Frame 2.90 - 3.40, Prediction: 0.45
Frame 3.00 - 3.50, Prediction: 0.45
Frame 3.10 - 3.60, Prediction: 0.49
Frame 3.20 - 3.70, Prediction: 0.44
Frame 3.30 - 3.80, Prediction: 0.45
Frame 3.40 - 3.90, Prediction: 0.48
Frame 3.50 - 4.00, Prediction: 0.51
Frame 3.60 - 4.10, Predictio